# MLP Experiment: Linear Combination of ReLU + Sigmoid

This notebook tests a dynamic activation function that is a **linear combination** of ReLU and Sigmoid:

$$f(x) = a \cdot \text{relu}(x) + b \cdot \text{sigmoid}(x)$$

where $a$ and $b$ are **learnable per-neuron parameters** (initialized to 0.5 each).

**Pipeline** (same as the Sigmoid experiment):
1. Train a **Baseline (ReLU)** MLP with standard fixed ReLU activations
2. Copy its trained weights, replace ReLU with `DynamicReLUSigmoid`, and **finetune only a, b** (weights frozen)

**Gradients** (mathematically correct backprop):
- $\frac{\partial f}{\partial a} = \text{relu}(x)$ → used to update $a$
- $\frac{\partial f}{\partial b} = \text{sigmoid}(x)$ → used to update $b$
- $\frac{\partial f}{\partial x} = a \cdot \text{relu}'(x) + b \cdot \sigma'(x)$ → backprop to previous layer

**Datasets**: MNIST, Fashion-MNIST, CIFAR-10 (20 seeds each)

**Learnable parameters \(a\) and \(b\)**

Each neuron in the DynamicReLUSigmoid activation has its own learnable scalars \(a\) and \(b\), initialized to **0.5**. During finetuning, **weights are frozen** and only \(a, b\) are updated by gradient descent.

The activation is:
\[
f(x) = a \cdot \text{ReLU}(x) + b \cdot \sigma(x)
\]

Gradients used for updates:
- \(\frac{\partial f}{\partial a} = \text{ReLU}(x)\)
- \(\frac{\partial f}{\partial b} = \sigma(x)\)

So, at each step:
\[
a \leftarrow a - \eta \cdot \frac{\partial \mathcal{L}}{\partial f} \cdot \text{ReLU}(x)
\qquad
b \leftarrow b - \eta \cdot \frac{\partial \mathcal{L}}{\partial f} \cdot \sigma(x)
\]

where \(\eta\) is the learning rate.

In [ ]:
import sys
import os

# Add src to path for imports
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import numpy as np
import pandas as pd
from datetime import datetime
from typing import List, Dict, Tuple
from dataclasses import dataclass, field
from scipy import stats

from data_utils import DatasetConfig, DataManager
from mlp_trainer import MLPTrainer, ExperimentResult, MLPExperiment
from mlp import MLP, MLPConfig, create_baseline_mlp, create_relu_sigmoid_mlp

print("Imports successful!")

Imports successful!


## Configuration

In [17]:
# Experiment Configuration
N_SEEDS = 20
SEEDS = [42, 123, 456, 789, 1001, 1111, 2222, 3333, 4444, 5555,
         6666, 7777, 8888, 9999, 1010, 2020, 3030, 4040, 5050, 6060]
HIDDEN_DIMS = [256, 128]  # Two hidden layers
EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 0.01
ACTIVATION_EPOCHS = 30

print("Configuration:")
print(f"  Number of seeds: {N_SEEDS}")
print(f"  Hidden layers: {HIDDEN_DIMS}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Activation training epochs: {ACTIVATION_EPOCHS}")

Configuration:
  Number of seeds: 20
  Hidden layers: [256, 128]
  Epochs: 30
  Batch size: 128
  Learning rate: 0.01
  Activation training epochs: 30


## Helper Classes and Functions

In [18]:
@dataclass
class AggregatedResult:
    """Aggregated results from multiple seeds."""
    model_name: str
    train_accs: List[float] = field(default_factory=list)
    test_accs: List[float] = field(default_factory=list)
    times: List[float] = field(default_factory=list)
    epochs: List[int] = field(default_factory=list)
    
    @property
    def train_mean(self) -> float:
        return np.mean(self.train_accs)
    
    @property
    def train_std(self) -> float:
        return np.std(self.train_accs)
    
    @property
    def test_mean(self) -> float:
        return np.mean(self.test_accs)
    
    @property
    def test_std(self) -> float:
        return np.std(self.test_accs)
    
    @property
    def time_mean(self) -> float:
        return np.mean(self.times)
    
    @property
    def epochs_mean(self) -> float:
        return np.mean(self.epochs)

In [19]:
def load_dataset(dataset_type: str, test_size: float = 0.2, random_state: int = 42):
    """Load and prepare dataset."""
    print(f"Loading {dataset_type} dataset...")
    config = DatasetConfig(
        dataset_type=dataset_type,
        test_size=test_size,
        random_state=random_state,
    )
    dm = DataManager(config)
    X_train, X_test, y_train, y_test = dm.generate_dataset()
    
    print(f"  Training samples: {len(X_train):,}")
    print(f"  Test samples: {len(X_test):,}")
    print(f"  Input features: {X_train.shape[1]}")
    print(f"  Classes: {len(np.unique(y_train))}")
    
    return X_train, X_test, y_train, y_test

In [32]:
def compute_significance(baseline_accs: List[float], model_accs: List[float], alpha: float = 0.05) -> Dict:
    """
    Run paired statistical tests comparing model vs baseline accuracy across seeds.
    Returns dict with t-test and Wilcoxon signed-rank test results.
    """
    baseline = np.array(baseline_accs)
    model = np.array(model_accs)
    diffs = model - baseline

    result = {
        "mean_diff": np.mean(diffs),
        "std_diff": np.std(diffs, ddof=1),
        "n": len(diffs),
        "alpha": alpha,
    }

    # Paired t-test (two-sided)
    t_stat, t_pval = stats.ttest_rel(model, baseline)
    result["t_stat"] = t_stat
    result["t_pval"] = t_pval
    result["t_significant"] = t_pval < alpha

    # Wilcoxon signed-rank test (non-parametric, two-sided)
    # Requires at least some non-zero differences
    if np.any(diffs != 0):
        w_stat, w_pval = stats.wilcoxon(model, baseline, alternative='two-sided')
        result["w_stat"] = w_stat
        result["w_pval"] = w_pval
        result["w_significant"] = w_pval < alpha
    else:
        result["w_stat"] = None
        result["w_pval"] = 1.0
        result["w_significant"] = False

    # Cohen's d (effect size)
    pooled_std = np.sqrt((np.std(baseline, ddof=1)**2 + np.std(model, ddof=1)**2) / 2)
    result["cohens_d"] = np.mean(diffs) / pooled_std if pooled_std > 0 else 0.0

    return result


def print_significance(sig: Dict) -> None:
    """Pretty-print statistical significance results."""
    print(f"\n📊 Statistical Significance Tests (α = {sig['alpha']}):")
    print(f"   Mean difference: {sig['mean_diff']:+.6f} ± {sig['std_diff']:.6f} (n={sig['n']})")
    print(f"   Cohen's d (effect size): {sig['cohens_d']:.4f}", end="")
    d = abs(sig['cohens_d'])
    if d < 0.2:
        print(" (negligible)")
    elif d < 0.5:
        print(" (small)")
    elif d < 0.8:
        print(" (medium)")
    else:
        print(" (large)")

    def get_verdict(is_sig, mean_diff):
        if not is_sig:
            return "➖ NOT significantly different"
        return "✅ SIGNIFICANTLY BETTER" if mean_diff > 0 else "❌ SIGNIFICANTLY WORSE"

    # Paired t-test
    verdict_t = get_verdict(sig['t_significant'], sig['mean_diff'])
    print(f"   Paired t-test:        t = {sig['t_stat']:+.4f},  p = {sig['t_pval']:.6f}  →  {verdict_t}")

    # Wilcoxon
    if sig['w_stat'] is not None:
        verdict_w = get_verdict(sig['w_significant'], sig['mean_diff'])
        print(f"   Wilcoxon signed-rank: W = {sig['w_stat']:.1f},    p = {sig['w_pval']:.6f}  →  {verdict_w}")
    else:
        print(f"   Wilcoxon signed-rank: N/A (all differences are zero)")


def print_results(agg_results: Dict[str, AggregatedResult], dataset_name: str, n_seeds: int) -> None:
    """Print comparison of aggregated experiment results with statistical significance."""
    print("\n" + "=" * 100)
    print(f"EXPERIMENT RESULTS: {dataset_name.upper()} ({n_seeds} seeds)")
    print("=" * 100)
    
    headers = ["Model", "Train Acc", "Test Acc", "Time (s)", "Epochs"]
    row_format = "{:<45} {:>18} {:>18} {:>10} {:>8}"
    
    print(row_format.format(*headers))
    print("-" * 100)
    
    for name, r in agg_results.items():
        print(row_format.format(
            name[:45],
            f"{r.train_mean:.4f} ± {r.train_std:.4f}",
            f"{r.test_mean:.4f} ± {r.test_std:.4f}",
            f"{r.time_mean:.2f}",
            f"{r.epochs_mean:.1f}",
        ))
    
    print("-" * 100)
    
    # Highlight winner
    best_name = max(agg_results.keys(), key=lambda x: agg_results[x].test_mean)
    best = agg_results[best_name]
    baseline = agg_results.get("Baseline (ReLU)", list(agg_results.values())[0])
    improvement = best.test_mean - baseline.test_mean
    
    print(f"\n🏆 Best: {best_name}")
    print(f"   Test Accuracy: {best.test_mean:.4f} ± {best.test_std:.4f}")
    print(f"   Improvement over baseline: {improvement:+.4f} ({improvement*100:+.2f}%)")

    # Statistical significance for each non-baseline model vs baseline
    baseline_key = "Baseline (ReLU)"
    if baseline_key in agg_results:
        for name, r in agg_results.items():
            if name == baseline_key:
                continue
            sig = compute_significance(
                agg_results[baseline_key].test_accs,
                r.test_accs,
            )
            print(f"\n--- {name} vs Baseline (ReLU) ---")
            print_significance(sig)

In [21]:
def save_results(all_results: dict, filepath: str, n_seeds: int) -> None:
    """Save aggregated results to CSV file, including statistical significance."""
    rows = []
    for dataset_name, agg_results in all_results.items():
        baseline_key = "Baseline (ReLU)"
        baseline_accs = agg_results[baseline_key].test_accs if baseline_key in agg_results else None

        for model_name, r in agg_results.items():
            row = {
                "dataset": dataset_name,
                "model_name": model_name,
                "n_seeds": n_seeds,
                "train_acc_mean": r.train_mean,
                "train_acc_std": r.train_std,
                "test_acc_mean": r.test_mean,
                "test_acc_std": r.test_std,
                "time_mean_seconds": r.time_mean,
                "epochs_mean": r.epochs_mean,
                "train_accs": str(r.train_accs),
                "test_accs": str(r.test_accs),
                "timestamp": datetime.now().isoformat(),
            }

            # Add significance columns for non-baseline models
            if model_name != baseline_key and baseline_accs is not None:
                sig = compute_significance(baseline_accs, r.test_accs)
                row["paired_ttest_pval"] = sig["t_pval"]
                row["paired_ttest_significant"] = sig["t_significant"]
                row["wilcoxon_pval"] = sig["w_pval"]
                row["wilcoxon_significant"] = sig["w_significant"]
                row["cohens_d"] = sig["cohens_d"]
                row["mean_diff"] = sig["mean_diff"]

            rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_csv(filepath, index=False)
    print(f"\nResults saved to: {filepath}")

## Experiment Functions

In [22]:
def run_relu_baseline(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    hidden_dims: List[int],
    epochs: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
    verbose: bool = False,
) -> Tuple[MLP, ExperimentResult]:
    """
    Train baseline ReLU MLP.
    Returns both the trained model and the experiment result.
    """
    import time
    from sklearn.metrics import accuracy_score
    
    np.random.seed(seed)
    
    input_dim = X_train.shape[1]
    output_dim = len(np.unique(y_train))
    
    model = create_baseline_mlp(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        output_dim=output_dim,
        learning_rate=learning_rate,
    )
    
    if verbose:
        print("\n" + "=" * 60)
        print("BASELINE MLP (ReLU)")
        print("=" * 60)
        print(model.summary())
    
    trainer = MLPTrainer(
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        verbose=verbose,
    )
    
    history = trainer.train(model, X_train, y_train, X_test, y_test)
    
    train_acc = trainer._compute_accuracy(model, X_train, y_train)
    test_acc = trainer._compute_accuracy(model, X_test, y_test)
    
    result = ExperimentResult(
        model_name="Baseline MLP (ReLU)",
        train_accuracy=train_acc,
        test_accuracy=test_acc,
        training_time=history.training_time,
        epochs_trained=history.epochs_trained,
        model_params=model.num_params,
    )
    
    return model, result

In [23]:
def run_relu_sigmoid_finetuning(
    baseline_model: MLP,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    activation_epochs: int,
    batch_size: int,
    learning_rate: float,
    per_neuron: bool = True,
    verbose: bool = False,
) -> ExperimentResult:
    """
    Take a trained baseline ReLU model, copy its weights into a model
    with DynamicReLUSigmoid activations: a*relu(x) + b*sigmoid(x),
    and train only the activation parameters a, b (weights frozen).
    """
    # Copy baseline weights into a model with DynamicReLUSigmoid activations
    dynamic_model = baseline_model.copy_with_relu_sigmoid_activations(
        per_neuron=per_neuron,
        activation_lr=learning_rate,
    )
    
    mode_str = "Per-Neuron" if per_neuron else "Per-Layer"
    
    if verbose:
        print("\n" + "=" * 60)
        print(f"RELU+SIGMOID ACTIVATION FINETUNING ({mode_str})")
        print("=" * 60)
        print("Activation: a*relu(x) + b*sigmoid(x)")
        print("Starting from trained baseline ReLU weights...")
        print(dynamic_model.summary())
    
    trainer = MLPTrainer(
        epochs=activation_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        verbose=verbose,
    )
    
    # Check accuracy before activation training
    pre_train_acc = trainer._compute_accuracy(dynamic_model, X_test, y_test)
    if verbose:
        print(f"\nTest accuracy before activation training: {pre_train_acc:.4f}")
        print(f"(a=0.5, b=0.5 -> 0.5*relu + 0.5*sigmoid, NOT the same as pure ReLU)")
        print(f"\n--- Training activation parameters a, b (weights frozen) ---")
    
    # Train only activation parameters (freeze weights)
    activation_history = trainer.train_activation_params(
        dynamic_model, X_train, y_train,
        epochs=activation_epochs,
        X_val=X_test, y_val=y_test
    )
    
    train_acc = trainer._compute_accuracy(dynamic_model, X_train, y_train)
    test_acc = trainer._compute_accuracy(dynamic_model, X_test, y_test)
    
    if verbose:
        print(f"\nActivation training complete.")
        print(f"Test accuracy: {test_acc:.4f} (was {pre_train_acc:.4f}, improvement: {test_acc - pre_train_acc:+.4f})")
    
    # Collect activation info
    activation_info = []
    for layer in dynamic_model.layers:
        if hasattr(layer, 'activation') and layer.activation.is_learnable:
            activation_info.append(layer.activation.info)
    
    return ExperimentResult(
        model_name=f"a*ReLU + b*Sigmoid Finetuning ({mode_str})",
        train_accuracy=train_acc,
        test_accuracy=test_acc,
        training_time=activation_history.training_time,
        epochs_trained=activation_history.epochs_trained,
        model_params=dynamic_model.num_params,
        activation_info="; ".join(activation_info),
    )

In [24]:
def run_single_seed(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
    hidden_dims: List[int],
    epochs: int,
    activation_epochs: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
    verbose: bool = False,
) -> Dict[str, ExperimentResult]:
    """
    Run all experiment variants for a single seed.
    
    1. Baseline ReLU
    2. Copy weights → DynamicReLUSigmoid finetuning (per-neuron)
    """
    results = {}
    
    # 1. Train Baseline ReLU MLP
    baseline_model, baseline_result = run_relu_baseline(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        hidden_dims=hidden_dims,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        seed=seed,
        verbose=verbose,
    )
    results["Baseline (ReLU)"] = baseline_result
    
    # 2. Copy baseline weights → finetune a*relu + b*sigmoid activations
    activation_result = run_relu_sigmoid_finetuning(
        baseline_model=baseline_model,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        activation_epochs=activation_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        per_neuron=True,
        verbose=verbose,
    )
    results["a*ReLU + b*Sigmoid Finetuning (Per-Neuron)"] = activation_result
    
    return results

In [25]:
def run_experiment_suite(
    dataset_type: str,
    hidden_dims: List[int],
    epochs: int,
    activation_epochs: int,
    batch_size: int,
    learning_rate: float,
    seeds: List[int],
) -> Dict[str, AggregatedResult]:
    """Run experiments with multiple seeds and aggregate results."""
    
    # Load data once (use first seed for train/test split)
    X_train, X_test, y_train, y_test = load_dataset(dataset_type, random_state=seeds[0])
    
    # Initialize aggregated results
    model_names = [
        "Baseline (ReLU)",
        "a*ReLU + b*Sigmoid Finetuning (Per-Neuron)",
    ]
    agg_results = {name: AggregatedResult(name) for name in model_names}
    
    # Run experiments for each seed
    for i, seed in enumerate(seeds):
        print(f"\n  Seed {i+1}/{len(seeds)} (seed={seed})...")
        
        seed_results = run_single_seed(
            X_train, X_test, y_train, y_test,
            hidden_dims=hidden_dims,
            epochs=epochs,
            activation_epochs=activation_epochs,
            batch_size=batch_size,
            learning_rate=learning_rate,
            seed=seed,
            verbose=False,
        )
        
        # Aggregate results
        for name, result in seed_results.items():
            agg_results[name].train_accs.append(result.train_accuracy)
            agg_results[name].test_accs.append(result.test_accuracy)
            agg_results[name].times.append(result.training_time)
            agg_results[name].epochs.append(result.epochs_trained)
        
        # Print progress
        baseline_acc = seed_results["Baseline (ReLU)"].test_accuracy
        finetuned_acc = seed_results["a*ReLU + b*Sigmoid Finetuning (Per-Neuron)"].test_accuracy
        print(f"    Baseline: {baseline_acc:.4f}, a*ReLU+b*Sigmoid: {finetuned_acc:.4f} (Δ={finetuned_acc-baseline_acc:+.4f})")
    
    return agg_results

## Run Experiments

In [26]:
print("=" * 100)
print("MLP EXPERIMENT: Baseline ReLU vs a*ReLU + b*Sigmoid Activation Finetuning")
print("Activation: f(x) = a * relu(x) + b * sigmoid(x), with learnable a, b per neuron")
print("Approach: Train baseline ReLU once, copy model, finetune activations on copy")
print("=" * 100)

all_results = {}

MLP EXPERIMENT: Baseline ReLU vs a*ReLU + b*Sigmoid Activation Finetuning
Activation: f(x) = a * relu(x) + b * sigmoid(x), with learnable a, b per neuron
Approach: Train baseline ReLU once, copy model, finetune activations on copy


### MNIST Dataset

In [27]:
print("\n" + "🌟" * 40)
print("DATASET: MNIST (Digits)")
print("🌟" * 40)

try:
    mnist_results = run_experiment_suite(
        dataset_type='mnist',
        hidden_dims=HIDDEN_DIMS,
        epochs=EPOCHS,
        activation_epochs=ACTIVATION_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        seeds=SEEDS,
    )
    all_results['mnist'] = mnist_results
    print_results(mnist_results, "MNIST", N_SEEDS)
except Exception as e:
    print(f"Skipping MNIST: {e}")


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
DATASET: MNIST (Digits)
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Loading mnist dataset...
  Training samples: 56,000
  Test samples: 14,000
  Input features: 784
  Classes: 10

  Seed 1/20 (seed=42)...
    Baseline: 0.9629, a*ReLU+b*Sigmoid: 0.9584 (Δ=-0.0046)

  Seed 2/20 (seed=123)...
    Baseline: 0.9635, a*ReLU+b*Sigmoid: 0.9593 (Δ=-0.0042)

  Seed 3/20 (seed=456)...
    Baseline: 0.9641, a*ReLU+b*Sigmoid: 0.9595 (Δ=-0.0046)

  Seed 4/20 (seed=789)...
    Baseline: 0.9651, a*ReLU+b*Sigmoid: 0.9611 (Δ=-0.0041)

  Seed 5/20 (seed=1001)...
    Baseline: 0.9642, a*ReLU+b*Sigmoid: 0.9596 (Δ=-0.0046)

  Seed 6/20 (seed=1111)...
    Baseline: 0.9640, a*ReLU+b*Sigmoid: 0.9592 (Δ=-0.0048)

  Seed 7/20 (seed=2222)...
    Baseline: 0.9641, a*ReLU+b*Sigmoid: 0.9596 (Δ=-0.0045)

  Seed 8/20 (seed=3333)...
    Baseline: 0.9646, a*ReLU+b*Sigmoid: 0.9605 (Δ=-0.0041)

  Seed 9/20 (seed=4444)...
    Baseline: 0.9639, a*ReLU+b*Sigmoid: 0.9576 (Δ=-0.0063)

  S

### Fashion-MNIST Dataset

In [28]:
print("\n" + "🌟" * 40)
print("DATASET: FASHION-MNIST")
print("🌟" * 40)

fashion_results = run_experiment_suite(
    dataset_type='fashion_mnist',
    hidden_dims=HIDDEN_DIMS,
    epochs=EPOCHS,
    activation_epochs=ACTIVATION_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seeds=SEEDS,
)
all_results['fashion_mnist'] = fashion_results
print_results(fashion_results, "Fashion-MNIST", N_SEEDS)


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
DATASET: FASHION-MNIST
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Loading fashion_mnist dataset...
  Training samples: 56,000
  Test samples: 14,000
  Input features: 784
  Classes: 10

  Seed 1/20 (seed=42)...
    Baseline: 0.8876, a*ReLU+b*Sigmoid: 0.8830 (Δ=-0.0046)

  Seed 2/20 (seed=123)...
    Baseline: 0.8849, a*ReLU+b*Sigmoid: 0.8803 (Δ=-0.0046)

  Seed 3/20 (seed=456)...
    Baseline: 0.8869, a*ReLU+b*Sigmoid: 0.8817 (Δ=-0.0051)

  Seed 4/20 (seed=789)...
    Baseline: 0.8859, a*ReLU+b*Sigmoid: 0.8807 (Δ=-0.0051)

  Seed 5/20 (seed=1001)...
    Baseline: 0.8839, a*ReLU+b*Sigmoid: 0.8830 (Δ=-0.0009)

  Seed 6/20 (seed=1111)...
    Baseline: 0.8821, a*ReLU+b*Sigmoid: 0.8804 (Δ=-0.0017)

  Seed 7/20 (seed=2222)...
    Baseline: 0.8834, a*ReLU+b*Sigmoid: 0.8804 (Δ=-0.0030)

  Seed 8/20 (seed=3333)...
    Baseline: 0.8837, a*ReLU+b*Sigmoid: 0.8830 (Δ=-0.0007)

  Seed 9/20 (seed=4444)...
    Baseline: 0.8874, a*ReLU+b*Sigmoid: 0.8828 (Δ=-0.004

### CIFAR-10 Dataset

In [29]:
print("\n" + "🌟" * 40)
print("DATASET: CIFAR-10 (Color Images)")
print("🌟" * 40)

cifar_results = run_experiment_suite(
    dataset_type='cifar10',
    hidden_dims=HIDDEN_DIMS,
    epochs=EPOCHS,
    activation_epochs=ACTIVATION_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seeds=SEEDS,
)
all_results['cifar10'] = cifar_results
print_results(cifar_results, "CIFAR-10", N_SEEDS)


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
DATASET: CIFAR-10 (Color Images)
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Loading cifar10 dataset...
CIFAR-10 loaded: 60000 samples, 3072 features
  Training samples: 48,000
  Test samples: 12,000
  Input features: 3072
  Classes: 10

  Seed 1/20 (seed=42)...
    Baseline: 0.5140, a*ReLU+b*Sigmoid: 0.5184 (Δ=+0.0044)

  Seed 2/20 (seed=123)...
    Baseline: 0.5056, a*ReLU+b*Sigmoid: 0.5162 (Δ=+0.0106)

  Seed 3/20 (seed=456)...
    Baseline: 0.5128, a*ReLU+b*Sigmoid: 0.5142 (Δ=+0.0014)

  Seed 4/20 (seed=789)...
    Baseline: 0.5122, a*ReLU+b*Sigmoid: 0.5198 (Δ=+0.0076)

  Seed 5/20 (seed=1001)...
    Baseline: 0.5148, a*ReLU+b*Sigmoid: 0.5172 (Δ=+0.0023)

  Seed 6/20 (seed=1111)...
    Baseline: 0.5150, a*ReLU+b*Sigmoid: 0.5191 (Δ=+0.0041)

  Seed 7/20 (seed=2222)...
    Baseline: 0.5144, a*ReLU+b*Sigmoid: 0.5226 (Δ=+0.0082)

  Seed 8/20 (seed=3333)...
    Baseline: 0.5152, a*ReLU+b*Sigmoid: 0.5251 (Δ=+0.0098)

  Seed 9/20 (seed=4444)...
    B

## Final Summary

In [33]:
print("\n" + "=" * 100)
print(f"FINAL SUMMARY (Mean ± Std over {N_SEEDS} seeds)")
print("=" * 100)

for dataset_name, agg_results in all_results.items():
    print(f"\n📊 {dataset_name.upper()}:")
    
    baseline = agg_results.get("Baseline (ReLU)")
    best_name = max(agg_results.keys(), key=lambda x: agg_results[x].test_mean)
    best = agg_results[best_name]
    improvement = best.test_mean - baseline.test_mean
    
    print(f"   Baseline (ReLU): {baseline.test_mean:.4f} ± {baseline.test_std:.4f}")
    print(f"   Best ({best_name}): {best.test_mean:.4f} ± {best.test_std:.4f}")
    print(f"   Improvement: {improvement:+.4f} ({improvement*100:+.2f}%)")

    # Statistical significance
    for name, r in agg_results.items():
        if name == "Baseline (ReLU)":
            continue
        sig = compute_significance(baseline.test_accs, r.test_accs)
        
        if sig['t_significant']:
            if sig['mean_diff'] > 0:
                verdict = "✅ significantly BETTER"
            else:
                verdict = "❌ significantly WORSE"
        else:
            verdict = "➖ NOT significantly different"
            
        print(f"   [{name}] vs Baseline:")
        print(f"      Paired t-test p-value: {sig['t_pval']:.6f} → {verdict} (Cohen's d = {sig['cohens_d']:.4f})")


FINAL SUMMARY (Mean ± Std over 20 seeds)

📊 MNIST:
   Baseline (ReLU): 0.9639 ± 0.0006
   Best (Baseline (ReLU)): 0.9639 ± 0.0006
   Improvement: +0.0000 (+0.00%)
   [a*ReLU + b*Sigmoid Finetuning (Per-Neuron)] vs Baseline:
      Paired t-test p-value: 0.000000 → ❌ significantly WORSE (Cohen's d = -5.4714)

📊 FASHION_MNIST:
   Baseline (ReLU): 0.8845 ± 0.0020
   Best (Baseline (ReLU)): 0.8845 ± 0.0020
   Improvement: +0.0000 (+0.00%)
   [a*ReLU + b*Sigmoid Finetuning (Per-Neuron)] vs Baseline:
      Paired t-test p-value: 0.000000 → ❌ significantly WORSE (Cohen's d = -1.5637)

📊 CIFAR10:
   Baseline (ReLU): 0.5155 ± 0.0038
   Best (a*ReLU + b*Sigmoid Finetuning (Per-Neuron)): 0.5213 ± 0.0036
   Improvement: +0.0058 (+0.58%)
   [a*ReLU + b*Sigmoid Finetuning (Per-Neuron)] vs Baseline:
      Paired t-test p-value: 0.000001 → ✅ significantly BETTER (Cohen's d = 1.5168)


## Save Results

In [ ]:
output_path = "mlp_relu_sigmoid_experiment_results.csv"
save_results(all_results, output_path, N_SEEDS)

# Display the saved results
df = pd.read_csv(output_path)
df